# 第六节课课件：面向 Burgers 方程的保守型 PINN

> 课程定位：从第五节的“线性平流方程守恒约束”正式迁移到论文主线：**Burgers 方程上的 Conservation-PINN**。

---


# 0. 本节课目标

本节课结束后，学生应该能够：

1. 说清楚线性平流方程与 Burgers 方程的关系；
2. 理解 Burgers 方程为什么是流体与非线性波动问题中的经典模型；
3. 写出 Burgers 方程的 PINN 残差；
4. 理解为什么周期边界下 Burgers 方程仍然具有质量守恒；
5. 把第五节的 `conservation_loss` 迁移到 Burgers 方程；
6. 设计 Baseline PINN 与 Conservation-PINN 的对比实验。

---


# 1. 从第五节课回顾：我们已经做了什么？

第五节课我们研究的是线性平流方程：

$$
u_t + c u_x = 0
$$

它可以写成守恒律形式：

$$
u_t + F(u)_x = 0, \qquad F(u)=cu
$$

在周期边界下：

$$
M(t)=\int_0^1 u(x,t)\,dx
$$

保持不变。

第五节课的核心思想是：

> 普通 PINN 只约束局部 PDE 残差，不一定严格保持全局守恒量。  
> Conservation-PINN 把守恒量误差也写进损失函数，使网络更重视物理结构。

---


# 2. 为什么第六节要讲 Burgers 方程？

Burgers 方程是流体力学和非线性 PDE 中非常经典的入门模型。它虽然比 Navier-Stokes 简单很多，但已经包含了几个重要特征：

- 非线性对流；
- 波形传播；
- 波形变陡；
- 粘性扩散；
- 守恒律结构；
- 长时间预测中的稳定性问题。

因此它非常适合作为本科生论文里的主算例。

一句话理解：

> 热传导方程主要展示扩散；线性平流方程主要展示平移；Burgers 方程开始展示“非线性流动”。

---


# 3. Burgers 方程的两种常见形式

## 3.1 无粘 Burgers 方程

无粘 Burgers 方程写成：

$$
u_t + u u_x = 0
$$

也可以写成守恒律形式：

$$
u_t + \left(\frac{u^2}{2}\right)_x = 0
$$

因为：

$$
\left(\frac{u^2}{2}\right)_x = u u_x
$$

所以两种写法等价。

其中通量函数为：

$$
F(u)=\frac{u^2}{2}
$$

## 3.2 粘性 Burgers 方程

为了教学和训练稳定性，本节课建议先使用粘性 Burgers 方程：

$$
u_t + u u_x = \nu u_{xx}
$$

也可写为残差形式：

$$
u_t + u u_x - \nu u_{xx}=0
$$

其中：

- $u(x,t)$：待求解变量；
- $u u_x$：非线性对流项；
- $\nu u_{xx}$：粘性扩散项；
- $\nu$：粘性系数。

课堂中可以取：

$$
\nu = \frac{0.01}{\pi}
$$

这是很多 PINN 入门例子里常用的设置。

---


# 4. Burgers 方程的物理直觉

## 4.1 线性平流：所有点以同一速度走

线性平流方程：

$$
u_t + c u_x = 0
$$

速度 $c$ 是常数。无论某处 $u$ 大还是小，传播速度都一样。

因此波形整体平移，形状基本不变。

## 4.2 Burgers：传播速度由自己决定

Burgers 方程：

$$
u_t + u u_x = 0
$$

可以理解为：局部传播速度就是 $u$ 本身。

如果某处 $u$ 大，它传播得更快；如果某处 $u$ 小，它传播得更慢。于是波形会逐渐变陡。

这就是 Burgers 方程比线性平流更接近真实流动问题的原因。

---


# 5. Burgers 方程的守恒结构

## 5.1 无粘形式下的质量守恒

从守恒律形式出发：

$$
u_t + F(u)_x = 0
$$

对空间区间 $[0,1]$ 积分：

$$
\frac{d}{dt}\int_0^1 u(x,t)\,dx
+
F(u(1,t))-F(u(0,t))=0
$$

若周期边界成立：

$$
u(0,t)=u(1,t)
$$

则：

$$
F(u(0,t))=F(u(1,t))
$$

因此：

$$
\frac{d}{dt}\int_0^1 u(x,t)\,dx=0
$$

即：

$$
M(t)=M(0)
$$

## 5.2 粘性 Burgers 下为什么质量仍守恒？

粘性 Burgers 方程：

$$
u_t + \left(\frac{u^2}{2}\right)_x = \nu u_{xx}
$$

对 $[0,1]$ 积分：

$$
\frac{d}{dt}\int_0^1 u\,dx
+
\left[\frac{u^2}{2}\right]_0^1
=
\nu [u_x]_0^1
$$

若采用周期边界，并且导数也周期一致：

$$
u(0,t)=u(1,t),\qquad u_x(0,t)=u_x(1,t)
$$

则两侧边界项都抵消：

$$
\frac{d}{dt}\int_0^1 u\,dx=0
$$

所以粘性 Burgers 方程在周期边界下仍然保持总质量守恒。

---


# 6. PINN 如何求解 Burgers 方程？

## 6.1 网络表示

仍然用神经网络表示未知解：

$$
u_\theta(x,t)\approx u(x,t)
$$

输入：

$$
(x,t)
$$

输出：

$$
u_\theta(x,t)
$$

## 6.2 自动微分

用 PyTorch 自动微分得到：

$$
u_t,\quad u_x,\quad u_{xx}
$$

## 6.3 PDE 残差

对于粘性 Burgers 方程：

$$
f_\theta(x,t)
=
(u_\theta)_t
+
u_\theta (u_\theta)_x
-
\nu (u_\theta)_{xx}
$$

训练时希望：

$$
f_\theta(x,t)\approx 0
$$

对应代码核心为：

```python
def pde_residual_burgers(model, x, t, nu):
    u = model(x, t)
    u_t = gradients(u, t)
    u_x = gradients(u, x)
    u_xx = gradients(u_x, x)
    f = u_t + u * u_x - nu * u_xx
    return f
```

---


# 7. 损失函数设计

## 7.1 普通 PINN

普通 PINN 总损失为：

$$
\mathcal L_{base}
=
\lambda_{ic}\mathcal L_{ic}
+
\lambda_{bc}\mathcal L_{bc}
+
\lambda_f\mathcal L_f
$$

其中：

$$
\mathcal L_{ic}
=
\frac{1}{N_{ic}}\sum_i
|u_\theta(x_i,0)-u_0(x_i)|^2
$$

$$
\mathcal L_{bc}
=
\frac{1}{N_{bc}}\sum_i
|u_\theta(0,t_i)-u_\theta(1,t_i)|^2
+
\frac{1}{N_{bc}}\sum_i
|(u_\theta)_x(0,t_i)-(u_\theta)_x(1,t_i)|^2
$$

$$
\mathcal L_f
=
\frac{1}{N_f}\sum_j |f_\theta(x_j,t_j)|^2
$$

## 7.2 Conservation-PINN

引入守恒量：

$$
M_\theta(t)=\int_0^1 u_\theta(x,t)\,dx
$$

理论守恒量：

$$
M_0=\int_0^1 u_0(x)\,dx
$$

守恒损失：

$$
\mathcal L_{cons}
=
\frac{1}{N_t}\sum_n
|M_\theta(t_n)-M_0|^2
$$

改进后的总损失：

$$
\mathcal L
=
\lambda_{ic}\mathcal L_{ic}
+
\lambda_{bc}\mathcal L_{bc}
+
\lambda_f\mathcal L_f
+
\lambda_{cons}\mathcal L_{cons}
$$

---


# 8. 本节课建议采用的初始条件和边界条件

为了避免一开始就进入激波问题，建议使用光滑周期初值：

$$
u(x,0)=1+0.5\sin(2\pi x)
$$

周期边界：

$$
u(0,t)=u(1,t)
$$

导数周期一致：

$$
u_x(0,t)=u_x(1,t)
$$

理论质量：

$$
M_0=\int_0^1 \left(1+0.5\sin(2\pi x)\right)dx=1
$$

这和第五节线性平流的设置保持一致，方便学生比较。

---


# 9. 和第五节代码相比，需要改哪里？

第五节线性平流的残差是：

```python
f = u_t + c * u_x
```

第六节 Burgers 残差改为：

```python
f = u_t + u * u_x - nu * u_xx
```

也就是说：

1. 需要多算一个二阶导 `u_xx`；
2. 需要加入非线性项 `u * u_x`；
3. 需要设置粘性系数 `nu`；
4. 守恒量损失 `conservation_loss` 可以基本复用；
5. 周期边界损失也可以继续复用。

---


# 10. 实验设计

## 10.1 对比模型

| 实验组 | 方法 | 目的 |
|---|---|---|
| A | Baseline PINN | 普通 PINN 基线 |
| B | Conservation-PINN | 检查守恒约束是否降低质量漂移 |
| C | 不同 $\lambda_{cons}$ | 研究守恒权重敏感性 |

## 10.2 建议参数

| 参数 | 建议值 | 说明 |
|---|---:|---|
| $N_{ic}$ | 128 | 初值点数量 |
| $N_{bc}$ | 128 | 边界点数量 |
| $N_f$ | 3000-8000 | Burgers 比线性平流更难，可适当增加 |
| $N_{cons,t}$ | 50 | 守恒约束时间切片 |
| $N_{cons,x}$ | 200 | 质量积分网格 |
| hidden_dim | 64 或 128 | 网络宽度 |
| num_hidden | 4 或 5 | 网络深度 |
| learning rate | 1e-3 | Adam 初始学习率 |
| $\lambda_{cons}$ | 0.1, 1, 10 | 做敏感性实验 |

---


# 11. 评价指标

## 11.1 预测误差

如果有参考解 $u_{ref}$，可用：

$$
E_{L2}
=
\frac{\|u_\theta-u_{ref}\|_2}{\|u_{ref}\|_2}
$$

如果没有解析解，可以用高精度 FDM/FVM 解作为参考解。

## 11.2 守恒量漂移

$$
E_{cons}(t)=|M_\theta(t)-M_0|
$$

可以统计：

- mean conservation error；
- max conservation error；
- final-time conservation error。

## 11.3 长时间外推稳定性

训练区间：

$$
t\in[0,1]
$$

外推区间：

$$
t\in[1,2]\quad \text{或}\quad t\in[1,3]
$$

观察：

- 波形是否明显漂移；
- 幅值是否衰减过快或爆炸；
- 守恒量是否持续偏离；
- Conservation-PINN 是否比 Baseline 更稳。

---


# 12. 论文图表规划

第六节课之后，论文主实验应逐渐形成以下图表：

## Figure 1：方法示意图

展示：

```text
IC / BC / PDE residual / Conservation residual
                ↓
          Total loss
                ↓
          Neural network u_theta(x,t)
```

## Figure 2：Burgers 解场对比

建议包含：

- 参考解；
- Baseline PINN 预测解；
- Conservation-PINN 预测解；
- 两种方法的误差图。

## Figure 3：守恒量漂移曲线

横轴：$t$  
纵轴：$|M_\theta(t)-M_0|$

比较：

- Baseline PINN；
- Conservation-PINN。

## Figure 4：长时间外推时间切片

选择：

$$
t=0,\ 0.5,\ 1.0,\ 1.5,\ 2.0
$$

比较预测曲线和参考曲线。

## Table 1：定量指标汇总

| 方法 | L2 error | mean cons error | max cons error | final cons error |
|---|---:|---:|---:|---:|
| Baseline PINN | | | | |
| Conservation-PINN | | | | |

---


# 13. 课堂中要特别提醒学生的问题

## 13.1 守恒量更好，不代表点态误差一定更小

Conservation loss 主要约束全局积分量。它可能显著降低质量漂移，但不一定让每个点的误差都下降。

因此评价时不能只看一个指标。

## 13.2 Burgers 比线性平流更难训练

原因：

- 非线性项 $u u_x$ 增加训练难度；
- 粘性系数较小时，解会出现更尖锐结构；
- 长时间预测中误差更容易累积；
- 损失权重更敏感。

## 13.3 如果训练失败，优先检查这些点

1. `u_xx` 是否正确计算；
2. `requires_grad=True` 是否设置；
3. `lambda_cons` 是否过大；
4. `N_f` 是否太少；
5. 学习率是否过高；
6. 边界条件是否真正周期一致。

---


# 14. 60 分钟课堂安排

## 0-8 分钟：回顾第五节

- 线性平流方程；
- 守恒量 $M(t)$；
- Conservation loss；
- 长时间预测对比。

## 8-22 分钟：Burgers 方程数学背景

- 无粘 Burgers；
- 粘性 Burgers；
- 守恒律形式；
- 非线性通量 $F(u)=u^2/2$。

## 22-35 分钟：PINN 残差修改

重点讲：

```python
f = u_t + u * u_x - nu * u_xx
```

并解释每一项的物理含义。

## 35-48 分钟：Conservation-PINN 实验设计

- Baseline vs Conservation；
- $\lambda_{cons}$ 敏感性；
- 训练区间和外推区间；
- 守恒量误差曲线。

## 48-56 分钟：论文图表规划

让学生知道每张图是为了回答什么问题。

## 56-60 分钟：布置课后任务

明确下一步要改代码、跑实验、整理结果。

---


# 15. 课后任务

## 配套实验代码

本节配套代码为：

```text
section6_conservation_pinn_burgers.py
```

该文件使用 `# %%` 分隔教学单元，可在 VS Code 中像 Notebook 一样逐单元运行，也可在终端完整运行：

```bash
cd lessons
python -m pip install -r requirements.txt

# 小规模冒烟测试，确认环境和流程正确
python section6_conservation_pinn_burgers.py --quick

# 正式对比实验，每个模型训练 3000 轮
python section6_conservation_pinn_burgers.py --epochs 3000 --lambda-cons 1
```

实验结果保存在 `results/section6_burgers/`，包括训练曲线、解场与误差图、质量漂移曲线、长时间外推切片、定量指标 CSV 和模型权重。

做守恒权重敏感性实验时，可分别运行：

```bash
python section6_conservation_pinn_burgers.py --epochs 3000 --lambda-cons 0.1
python section6_conservation_pinn_burgers.py --epochs 3000 --lambda-cons 1
python section6_conservation_pinn_burgers.py --epochs 3000 --lambda-cons 10
```

不同权重的结果会自动保存到不同的 `results/section6_burgers_lambda_*/` 目录，避免相互覆盖。

## 必做

1. 复制第五节 Notebook，命名为 `section6_conservation_pinn_burgers.ipynb`。
2. 把 PDE 残差从线性平流改为 Burgers：

```python
f = u_t + u * u_x - nu * u_xx
```

3. 保留周期边界损失和守恒量损失。
4. 跑 Baseline PINN 与 Conservation-PINN。
5. 输出以下图：
   - 训练 loss 曲线；
   - 解场误差图；
   - 守恒量漂移曲线；
   - 长时间外推切片图。

## 选做

1. 比较 $\lambda_{cons}=0.1,1,10$；
2. 比较不同粘性系数 $\nu$；
3. 尝试 Adam + LBFGS 两阶段训练；
4. 用 FDM 生成参考解，而不是只依赖解析形式。

---


# 16. 一句话总结

第五节课让学生知道“守恒量可以写进 loss”。  
第六节课要让学生完成关键迁移：

> 从线性守恒律迁移到 Burgers 非线性守恒律，开始形成论文主实验。


# 配套实验：可逐单元运行的完整代码

下面的代码单元与讲义内容配套。首次运行建议保留 `RUN_QUICK = True`，确认环境和流程正确后再运行正式实验。


## 实验单元：Imports and configuration


In [ ]:
from __future__ import annotations

import argparse
import csv
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn


@dataclass
class Config:
    nu: float = 0.01 / np.pi
    train_t_max: float = 1.0
    test_t_max: float = 2.0
    epochs: int = 3000
    learning_rate: float = 1e-3
    n_ic: int = 128
    n_bc: int = 128
    n_f: int = 3000
    n_cons_t: int = 50
    n_cons_x: int = 200
    hidden_dim: int = 64
    num_hidden: int = 4
    lambda_ic: float = 10.0
    lambda_bc: float = 1.0
    lambda_f: float = 1.0
    lambda_cons: float = 1.0
    seed: int = 42
    print_every: int = 100
    ref_nx: int = 256
    ref_dt: float = 5e-4
    output_dir: str = "results/section6_burgers"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def initial_condition(x):
    """Smooth periodic initial condition with exact mass M0 = 1."""
    if isinstance(x, torch.Tensor):
        return 1.0 + 0.5 * torch.sin(2.0 * torch.pi * x)
    return 1.0 + 0.5 * np.sin(2.0 * np.pi * x)


## 实验单元：A periodic pseudo-spectral reference solver


In [ ]:
def solve_reference(
    times: np.ndarray, nu: float, nx: int = 256, dt: float = 5e-4
) -> tuple[np.ndarray, np.ndarray]:
    """Solve viscous Burgers on [0, 1) using spectral derivatives and RK4."""
    times = np.asarray(times, dtype=np.float64)
    if times.ndim != 1 or np.any(np.diff(times) < 0):
        raise ValueError("times must be a sorted one-dimensional array")

    x = np.arange(nx, dtype=np.float64) / nx
    wave_numbers = 2.0 * np.pi * np.fft.fftfreq(nx, d=1.0 / nx)
    cutoff = nx // 3
    dealias_mask = np.abs(np.fft.fftfreq(nx) * nx) <= cutoff
    u = initial_condition(x)

    def rhs(values: np.ndarray) -> np.ndarray:
        u_hat = np.fft.fft(values)
        flux_hat = np.fft.fft(0.5 * values**2)
        flux_hat[~dealias_mask] = 0.0
        flux_x = np.fft.ifft(1j * wave_numbers * flux_hat).real
        u_xx = np.fft.ifft(-(wave_numbers**2) * u_hat).real
        return -flux_x + nu * u_xx

    snapshots = np.empty((len(times), nx), dtype=np.float64)
    current_t = 0.0
    for index, target_t in enumerate(times):
        while current_t < target_t - 1e-14:
            step = min(dt, target_t - current_t)
            k1 = rhs(u)
            k2 = rhs(u + 0.5 * step * k1)
            k3 = rhs(u + 0.5 * step * k2)
            k4 = rhs(u + step * k3)
            u = u + step * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
            current_t += step
        snapshots[index] = u
    return x, snapshots


## 实验单元：PINN model and automatic differentiation


In [ ]:
class PINN(nn.Module):
    def __init__(self, hidden_dim: int = 64, num_hidden: int = 4):
        super().__init__()
        layers: list[nn.Module] = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(num_hidden - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.Tanh()])
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([x, t], dim=1))


def gradient(y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return torch.autograd.grad(
        y,
        x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
    )[0]


def burgers_residual(
    model: PINN, x: torch.Tensor, t: torch.Tensor, nu: float
) -> torch.Tensor:
    """f = u_t + u u_x - nu u_xx."""
    u = model(x, t)
    u_t = gradient(u, t)
    u_x = gradient(u, x)
    u_xx = gradient(u_x, x)
    return u_t + u * u_x - nu * u_xx


def periodic_bc_loss(model: PINN, t: torch.Tensor) -> torch.Tensor:
    """Enforce both u(0,t)=u(1,t) and u_x(0,t)=u_x(1,t)."""
    x_left = torch.zeros_like(t, requires_grad=True)
    x_right = torch.ones_like(t, requires_grad=True)
    u_left = model(x_left, t)
    u_right = model(x_right, t)
    ux_left = gradient(u_left, x_left)
    ux_right = gradient(u_right, x_right)
    return torch.mean((u_left - u_right) ** 2) + torch.mean(
        (ux_left - ux_right) ** 2
    )


def conservation_loss(
    model: PINN, times: torch.Tensor, n_x: int, target_mass: float = 1.0
) -> torch.Tensor:
    """Approximate M(t)=integral_0^1 u(x,t) dx on uniform periodic points."""
    x = torch.arange(n_x, device=times.device, dtype=times.dtype) / n_x
    x = x.reshape(1, -1).expand(len(times), -1).reshape(-1, 1)
    t = times.reshape(-1, 1).expand(-1, n_x).reshape(-1, 1)
    mass = model(x, t).reshape(len(times), n_x).mean(dim=1)
    return torch.mean((mass - target_mass) ** 2)


## 实验单元：Training


In [ ]:
def train_model(
    name: str, config: Config, device: torch.device, lambda_cons: float
) -> tuple[PINN, dict[str, list[float]]]:
    set_seed(config.seed)
    model = PINN(config.hidden_dim, config.num_hidden).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    history = {key: [] for key in ["total", "ic", "bc", "pde", "cons"]}

    for epoch in range(1, config.epochs + 1):
        optimizer.zero_grad()

        x_ic = torch.rand(config.n_ic, 1, device=device)
        t_ic = torch.zeros_like(x_ic)
        loss_ic = torch.mean((model(x_ic, t_ic) - initial_condition(x_ic)) ** 2)

        t_bc = config.train_t_max * torch.rand(config.n_bc, 1, device=device)
        loss_bc = periodic_bc_loss(model, t_bc)

        x_f = torch.rand(config.n_f, 1, device=device, requires_grad=True)
        t_f = (
            config.train_t_max
            * torch.rand(config.n_f, 1, device=device, requires_grad=True)
        )
        residual = burgers_residual(model, x_f, t_f, config.nu)
        loss_pde = torch.mean(residual**2)

        t_cons = config.train_t_max * torch.rand(config.n_cons_t, 1, device=device)
        loss_cons = conservation_loss(model, t_cons, config.n_cons_x)

        loss = (
            config.lambda_ic * loss_ic
            + config.lambda_bc * loss_bc
            + config.lambda_f * loss_pde
            + lambda_cons * loss_cons
        )
        loss.backward()
        optimizer.step()

        values = [loss, loss_ic, loss_bc, loss_pde, loss_cons]
        for key, value in zip(history, values):
            history[key].append(float(value.detach().cpu()))

        if epoch == 1 or epoch % config.print_every == 0:
            print(
                f"{name:>13s} | epoch {epoch:5d}/{config.epochs} | "
                f"loss={history['total'][-1]:.3e} | "
                f"pde={history['pde'][-1]:.3e} | "
                f"cons={history['cons'][-1]:.3e}"
            )
    return model, history


## 实验单元：Evaluation helpers


In [ ]:
@torch.no_grad()
def predict(
    model: PINN, x: np.ndarray, times: np.ndarray, device: torch.device
) -> np.ndarray:
    xx, tt = np.meshgrid(x, times)
    inputs_x = torch.tensor(xx.reshape(-1, 1), dtype=torch.float32, device=device)
    inputs_t = torch.tensor(tt.reshape(-1, 1), dtype=torch.float32, device=device)
    values = model(inputs_x, inputs_t).cpu().numpy()
    return values.reshape(len(times), len(x))


def metrics(prediction: np.ndarray, reference: np.ndarray) -> dict[str, float]:
    mass_error = np.abs(prediction.mean(axis=1) - 1.0)
    return {
        "relative_l2": float(np.linalg.norm(prediction - reference) / np.linalg.norm(reference)),
        "mean_cons_error": float(mass_error.mean()),
        "max_cons_error": float(mass_error.max()),
        "final_cons_error": float(mass_error[-1]),
    }


def save_metrics(rows: list[dict[str, float | str]], output_dir: Path) -> None:
    fieldnames = [
        "method",
        "relative_l2",
        "mean_cons_error",
        "max_cons_error",
        "final_cons_error",
    ]
    with (output_dir / "metrics.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


## 实验单元：Figures required by the lesson


In [ ]:
def plot_training(
    histories: dict[str, dict[str, list[float]]], output_dir: Path
) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, history in histories.items():
        axes[0].semilogy(history["total"], label=name)
        axes[1].semilogy(history["cons"], label=name)
    axes[0].set(title="Total training loss", xlabel="epoch", ylabel="loss")
    axes[1].set(title="Mass loss during training", xlabel="epoch", ylabel="loss")
    for axis in axes:
        axis.grid(alpha=0.3)
        axis.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "01_training_loss.png", dpi=180)
    plt.close(fig)


def plot_solution_fields(
    x: np.ndarray,
    times: np.ndarray,
    reference: np.ndarray,
    predictions: dict[str, np.ndarray],
    output_dir: Path,
) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True)
    extent = [x[0], x[-1], times[-1], times[0]]
    fields = [reference, predictions["Baseline"], predictions["Conservation"]]
    titles = ["Reference", "Baseline PINN", "Conservation-PINN"]
    vmax = max(np.max(np.abs(field - reference)) for field in fields[1:])
    for column, (field, title) in enumerate(zip(fields, titles)):
        image = axes[0, column].imshow(field, aspect="auto", extent=extent, cmap="viridis")
        axes[0, column].set_title(title)
        fig.colorbar(image, ax=axes[0, column])
        error = np.abs(field - reference)
        image = axes[1, column].imshow(
            error, aspect="auto", extent=extent, cmap="magma", vmin=0.0, vmax=vmax
        )
        axes[1, column].set_title(f"{title} absolute error")
        fig.colorbar(image, ax=axes[1, column])
    for axis in axes[:, 0]:
        axis.set_ylabel("t")
    for axis in axes[-1]:
        axis.set_xlabel("x")
    fig.tight_layout()
    fig.savefig(output_dir / "02_solution_and_error.png", dpi=180)
    plt.close(fig)


def plot_mass_drift(
    times: np.ndarray, predictions: dict[str, np.ndarray], output_dir: Path
) -> None:
    fig, axis = plt.subplots(figsize=(7, 4))
    for name, prediction in predictions.items():
        axis.semilogy(times, np.abs(prediction.mean(axis=1) - 1.0) + 1e-16, label=name)
    axis.axvline(1.0, color="black", linestyle="--", linewidth=1, label="training limit")
    axis.set(xlabel="t", ylabel="|M(t) - M(0)|", title="Mass conservation error")
    axis.grid(alpha=0.3)
    axis.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "03_mass_drift.png", dpi=180)
    plt.close(fig)


def plot_time_slices(
    x: np.ndarray,
    times: np.ndarray,
    reference: np.ndarray,
    predictions: dict[str, np.ndarray],
    output_dir: Path,
) -> None:
    requested = [0.0, 0.5, 1.0, 1.5, 2.0]
    indices = [int(np.argmin(np.abs(times - value))) for value in requested]
    fig, axes = plt.subplots(1, len(indices), figsize=(18, 3.5), sharey=True)
    for axis, index in zip(axes, indices):
        axis.plot(x, reference[index], color="black", linewidth=2, label="Reference")
        for name, prediction in predictions.items():
            axis.plot(x, prediction[index], linestyle="--", label=name)
        axis.set_title(f"t = {times[index]:.1f}")
        axis.set_xlabel("x")
        axis.grid(alpha=0.3)
    axes[0].set_ylabel("u(x,t)")
    axes[-1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(output_dir / "04_time_slices.png", dpi=180)
    plt.close(fig)


## 实验单元：Complete experiment


In [ ]:
def run_experiment(config: Config) -> None:
    set_seed(config.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    output_dir = Path.cwd() / config.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"device: {device}")
    print(f"results: {output_dir}")

    baseline, baseline_history = train_model("Baseline", config, device, lambda_cons=0.0)
    conservation, conservation_history = train_model(
        "Conservation", config, device, lambda_cons=config.lambda_cons
    )

    times = np.linspace(0.0, config.test_t_max, 101)
    x, reference = solve_reference(times, config.nu, config.ref_nx, config.ref_dt)
    predictions = {
        "Baseline": predict(baseline, x, times, device),
        "Conservation": predict(conservation, x, times, device),
    }

    rows: list[dict[str, float | str]] = []
    for name, prediction in predictions.items():
        row: dict[str, float | str] = {"method": name}
        row.update(metrics(prediction, reference))
        rows.append(row)
        print(name, {key: f"{value:.3e}" for key, value in row.items() if key != "method"})

    save_metrics(rows, output_dir)
    plot_training(
        {"Baseline": baseline_history, "Conservation": conservation_history}, output_dir
    )
    plot_solution_fields(x, times, reference, predictions, output_dir)
    plot_mass_drift(times, predictions, output_dir)
    plot_time_slices(x, times, reference, predictions, output_dir)
    torch.save(baseline.state_dict(), output_dir / "baseline_pinn.pt")
    torch.save(conservation.state_dict(), output_dir / "conservation_pinn.pt")


## 运行实验


In [ ]:
RUN_QUICK = True

cfg = Config()
if RUN_QUICK:
    cfg.epochs = 10
    cfg.n_ic = 32
    cfg.n_bc = 32
    cfg.n_f = 128
    cfg.n_cons_t = 8
    cfg.n_cons_x = 32
    cfg.hidden_dim = 32
    cfg.num_hidden = 3
    cfg.print_every = 1
    cfg.ref_nx = 128
    cfg.ref_dt = 1e-3
    cfg.output_dir = "results/section6_burgers_quick"

run_experiment(cfg)
